# Pancreatic Cyst Segmentation — Approach A (nnUNet) on Colab T4

**GPU:** T4 (15 GB VRAM)  
**Session limit:** ~12 hours — training resumes automatically across sessions via `--c`.

## Workflow
1. Mount Google Drive (stores dataset + checkpoints between sessions)
2. Clone/pull repo
3. Install dependencies
4. Prepare dataset (first time only)
5. Create T4-optimised plans (first time only)
6. Train — auto-resumes if checkpoint exists

In [ ]:
# ── 0. Verify GPU ─────────────────────────────────────────────────────────────
!nvidia-smi | head -15

In [ ]:
# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
# Google Drive is used to persist: dataset, preprocessed data, and checkpoints
# between Colab sessions (Colab disk resets on every reconnect).
#
# Expected Drive layout after first setup:
#   MyDrive/pancreas_cyst/
#     data/images/*.nii.gz
#     data/masks/cyst_*.nii.gz
#     nnUNet_preprocessed/   (generated once, ~6 GB)
#     nnUNet_results/        (checkpoints saved here)

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/pancreas_cyst'
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

In [ ]:
# ── 2. Clone or update repo ───────────────────────────────────────────────────
import os

REPO_DIR = '/content/Pancreas_cyst_segmentation'
REPO_URL = 'git@github.com:quannguyenai/Pancreas_cyst_segmentation.git'
# Or use HTTPS (no SSH key needed):
# REPO_URL = 'https://github.com/quannguyenai/Pancreas_cyst_segmentation.git'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# ── 3. Install dependencies ───────────────────────────────────────────────────
# Colab T4 already has PyTorch; we only need nnunetv2 + extras.
!pip install -q nnunetv2 monai nibabel SimpleITK pyyaml medpy
print('Dependencies installed.')

In [ ]:
# ── 4. Set environment variables ──────────────────────────────────────────────
import os

REPO_DIR   = '/content/Pancreas_cyst_segmentation'
DRIVE_ROOT = '/content/drive/MyDrive/pancreas_cyst'

# nnUNet reads these three env vars
os.environ['nnUNet_raw']          = f'{DRIVE_ROOT}/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = f'{DRIVE_ROOT}/nnUNet_preprocessed'
os.environ['nnUNet_results']      = f'{DRIVE_ROOT}/nnUNet_results'
os.environ['PANCREAS_CYST_ROOT']  = REPO_DIR

for var in ['nnUNet_raw', 'nnUNet_preprocessed', 'nnUNet_results']:
    os.makedirs(os.environ[var], exist_ok=True)
    print(f'{var} = {os.environ[var]}')

In [ ]:
# ── 5a. Update split CSVs and build nnUNet raw dataset (FIRST SESSION ONLY) ──
# Skip this cell if you have already run it and Drive has nnUNet_raw/Dataset001

import os
REPO_DIR   = '/content/Pancreas_cyst_segmentation'
DRIVE_ROOT = '/content/drive/MyDrive/pancreas_cyst'

raw_dataset = f"{os.environ['nnUNet_raw']}/Dataset001_PancreasCyst"
if os.path.exists(raw_dataset):
    print('Dataset001 already exists in Drive, skipping.')
else:
    # Update paths in split CSVs to point to Drive
    !python {REPO_DIR}/data/prepare_dataset.py \
        --config {REPO_DIR}/configs/paths.yaml \
        --update-txts \
        --build-nnunet
    print('Dataset prepared.')

In [ ]:
# ── 5b. Preprocess (FIRST SESSION ONLY, takes ~20-40 min) ────────────────────
import os, pathlib

preprocessed_dataset = pathlib.Path(os.environ['nnUNet_preprocessed']) / 'Dataset001_PancreasCyst'
plans_file = preprocessed_dataset / 'nnUNetPlans.json'

if plans_file.exists():
    print('Preprocessing already done, skipping.')
else:
    !nnUNetv2_plan_and_preprocess -d 1 --verify_dataset_integrity -np 2
    print('Preprocessing complete.')

In [ ]:
# ── 5c. Create T4-optimised plans (FIRST SESSION ONLY) ───────────────────────
# Reduces patch [40,192,256] → [40,128,192] to fit in T4's 15 GB VRAM.
import os, pathlib

REPO_DIR = '/content/Pancreas_cyst_segmentation'
t4_plans = (pathlib.Path(os.environ['nnUNet_preprocessed'])
            / 'Dataset001_PancreasCyst' / 'nnUNetPlans_T4.json')

if t4_plans.exists():
    print('T4 plans already exist, skipping.')
else:
    preprocessed_dir = str(pathlib.Path(os.environ['nnUNet_preprocessed']) / 'Dataset001_PancreasCyst')
    !python {REPO_DIR}/colab/create_t4_plans.py \
        --preprocessed-dir "{preprocessed_dir}"
    print('T4 plans created.')

In [ ]:
# ── 6. Train (auto-resumes if checkpoint exists) ──────────────────────────────
# Training runs for ~12 h per session; use --c flag to resume in subsequent sessions.
# Checkpoints are saved to Google Drive automatically.
#
# 1000 epochs × ~8-12 min/epoch on T4 ≈ 130-200 h total
# → expect 12-18 Colab sessions to complete full training
#
# nnUNet saves checkpoint_latest.pth every epoch — if session times out,
# the next session will resume from where it left off via --c.

import os, pathlib

FOLD       = 0
PLANS      = 'nnUNetPlans_T4'
DATASET_ID = 1
CONFIG     = '3d_fullres'

# Check if a checkpoint exists to decide whether to add --c
results_dir = (pathlib.Path(os.environ['nnUNet_results'])
               / 'Dataset001_PancreasCyst'
               / f'nnUNetTrainer__{PLANS}__{CONFIG}'
               / f'fold_{FOLD}')

resume_flag = '--c' if (results_dir / 'checkpoint_latest.pth').exists() else ''
print(f'Checkpoint exists: {bool(resume_flag)}')
print(f'Results dir: {results_dir}')

!nnUNetv2_train {DATASET_ID} {CONFIG} {FOLD} -p {PLANS} --npz {resume_flag}

In [ ]:
# ── 7. Check training progress ────────────────────────────────────────────────
import os, pathlib, json

FOLD   = 0
PLANS  = 'nnUNetPlans_T4'
CONFIG = '3d_fullres'

progress_png = (pathlib.Path(os.environ['nnUNet_results'])
                / 'Dataset001_PancreasCyst'
                / f'nnUNetTrainer__{PLANS}__{CONFIG}'
                / f'fold_{FOLD}'
                / 'progress.png')

if progress_png.exists():
    from IPython.display import Image
    display(Image(str(progress_png)))
else:
    print('No progress plot yet — training has not started or completed an epoch.')